In [1]:
import io
import ssl
import urllib.request
import pandas as pd

def load_df(url: str) -> pd.DataFrame:
    context = ssl._create_unverified_context()
    with urllib.request.urlopen(url, context=context) as response:
        return pd.read_csv(io.BytesIO(response.read()))

In [2]:
url = "https://raw.githubusercontent.com/prashant-kikani/breast-cancer-detection/refs/heads/master/breast-cancer-data.csv"
df = load_df(url)
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 569 entries, 0 to 568
Data columns (total 33 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   id                       569 non-null    int64  
 1   diagnosis                569 non-null    str    
 2   radius_mean              569 non-null    float64
 3   texture_mean             569 non-null    float64
 4   perimeter_mean           569 non-null    float64
 5   area_mean                569 non-null    float64
 6   smoothness_mean          569 non-null    float64
 7   compactness_mean         569 non-null    float64
 8   concavity_mean           569 non-null    float64
 9   concave points_mean      569 non-null    float64
 10  symmetry_mean            569 non-null    float64
 11  fractal_dimension_mean   569 non-null    float64
 12  radius_se                569 non-null    float64
 13  texture_se               569 non-null    float64
 14  perimeter_se             569 non-null

In [3]:
# Handling Nan Values
df.drop(columns=["Unnamed: 32"], inplace=True)
print(df.shape)

(569, 32)


In [4]:
X = df.drop(columns=['diagnosis'])
y = df[["diagnosis"]]

print("X shape:", X.shape, type(X), "y shape:", y.shape, type(y))

X shape: (569, 31) <class 'pandas.DataFrame'> y shape: (569, 1) <class 'pandas.DataFrame'>


In [5]:
# preprocessing
from sklearn.preprocessing import LabelEncoder

# Label Encoder
le = LabelEncoder()
y = le.fit_transform(df['diagnosis'])

In [6]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X = scaler.fit_transform(X)

In [7]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

print(X_train.shape)
print(X_test.shape)
print(y_train.shape)
print(y_test.shape)

(455, 31)
(114, 31)
(455,)
(114,)


In [8]:
import numpy as np

from trueml.linear_model import LinearModel
from trueml.losses import MSELoss

batch_size, n_features = X.shape
model = LinearModel(lr=0.0001, n_features=n_features, uops=True)
loss_fn = MSELoss(uops=True)

for epoch in range(10):
    y_pred = model.forward(X_train)

    dl_dy_pred = loss_fn.grad(y_train, y_pred)

    model.backward(dl_dy_pred) # gradient explosion/overflow: scaling

In [11]:
from trueml.viz import ComputationGraph

g = ComputationGraph()

g.build(model.bias)

g.view(open_browser=True)

In [10]:
eval_y = model.predict(X_test)


i = (eval_y >= 0.5).astype(int)

eval_y.shape

TypeError: '>=' not supported between instances of 'Tensor' and 'float'

In [ ]:
from sklearn.metrics import accuracy_score

print("Accuracy:", accuracy_score(y_test, i))

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

print("Accuracy :", accuracy_score(y_test, i))
print("Precision:", precision_score(y_test, i))
print("Recall   :", recall_score(y_test, i))
print("F1 Score :", f1_score(y_test, i))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, i))

print("\nClassification Report:")
print(classification_report(y_test, i))